In [2]:
!pip install -q transformers datasets seqeval accelerate

In [4]:
from datasets import load_dataset

raw = load_dataset("tomaarsen/conll2003")
print(raw)

label_list = raw["train"].features["ner_tags"].feature.names
print(label_list)
print(raw["train"][0])

README.md:   0%|          | 0.00/15.0k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/13.5k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.24MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  316kB            

...

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/14041 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3250 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3453 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'document_id', 'sentence_id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
{'id': '0', 'document_id': 1, 'sentence_id': 0, 'tokens': ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.'], 'pos_tags': [22, 42, 16, 21, 35, 37, 16, 21, 7], 'chunk_tags': [11, 21, 11, 12, 21, 22, 11, 12, 0], 'ner_tags': [3, 0, 7, 0, 0, 0, 7, 0, 0]}


In [5]:
from transformers import AutoTokenizer

model_name = "distilbert-base-cased"   # cased matters: capital letters signal names
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_and_align(examples):
    tokenized = tokenizer(examples["tokens"], truncation=True,
                          is_split_into_words=True)
    all_labels = []
    for i, tags in enumerate(examples["ner_tags"]):
        word_ids = tokenized.word_ids(batch_index=i)
        prev = None
        label_ids = []
        for w in word_ids:
            if w is None:            # special tokens like [CLS], [SEP]
                label_ids.append(-100)
            elif w != prev:          # first piece of a word
                label_ids.append(tags[w])
            else:                    # later pieces of the same word
                label_ids.append(-100)
            prev = w
        all_labels.append(label_ids)
    tokenized["labels"] = all_labels
    return tokenized

tokenized_ds = raw.map(tokenize_and_align, batched=True)

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

Map:   0%|          | 0/14041 [00:00<?, ? examples/s]

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

Map:   0%|          | 0/3453 [00:00<?, ? examples/s]

In [6]:
from transformers import AutoModelForTokenClassification

id2label = {i: l for i, l in enumerate(label_list)}
label2id = {l: i for i, l in enumerate(label_list)}

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  263MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)

    true_preds, true_labels = [], []
    for pred_row, label_row in zip(preds, labels):
        p_tags, l_tags = [], []
        for p, l in zip(pred_row, label_row):
            if l != -100:
                p_tags.append(label_list[p])
                l_tags.append(label_list[l])
        true_preds.append(p_tags)
        true_labels.append(l_tags)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

In [9]:
from transformers import TrainingArguments, Trainer, DataCollatorForTokenClassification

args = TrainingArguments(
    output_dir="ner-distilbert",
    eval_strategy="epoch",          # older versions call this evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_ds["train"],
    eval_dataset=tokenized_ds["validation"],
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.196307,0.056083,0.901235,0.909122,0.905161
2,0.038753,0.046137,0.926850,0.931841,0.929339
3,0.021510,0.043954,0.930792,0.937058,0.933915


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2634, training_loss=0.06667046458230565, metrics={'train_runtime': 134.8113, 'train_samples_per_second': 312.459, 'train_steps_per_second': 19.538, 'total_flos': 525319502290632.0, 'train_loss': 0.06667046458230565, 'epoch': 3.0})

In [10]:
results = trainer.evaluate(tokenized_ds["test"])
print(results)

Training Loss,Validation Loss,Epoch,Precision,Recall,F1
0.021510,0.108880,3,0.883008,0.898017,0.890449


{'eval_loss': 0.10887987166643143, 'eval_precision': 0.883008356545961, 'eval_recall': 0.8980169971671388, 'eval_f1': 0.8904494382022472}


In [11]:
trainer.save_model("ner-model")
tokenizer.save_pretrained("ner-model")

from transformers import pipeline
ner = pipeline("token-classification", model="ner-model",
               aggregation_strategy="simple")

text = "Ahmad works at Petronas in Kuala Lumpur."
for ent in ner(text):
    print(ent["entity_group"], "->", ent["word"], round(float(ent["score"]), 2))

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

PER -> Ahmad 0.97
ORG -> Petronas 0.99
LOC -> Kuala Lumpur 1.0


In [13]:
from google.colab import drive
drive.mount('/content/drive')
!cp -r ner-model /content/drive/MyDrive/ner-model

Mounted at /content/drive


In [14]:
from transformers import pipeline

ner = pipeline("token-classification", model="ner-model",
               aggregation_strategy="simple")

KEEP = {"PER", "LOC", "ORG"}   # ignore MISC

def detect(text):
    return [e for e in ner(text) if e["entity_group"] in KEEP]

def redact(text):
    ents = sorted(detect(text), key=lambda e: e["start"], reverse=True)
    for e in ents:                       # go backwards so positions stay valid
        text = text[:e["start"]] + f"[{e['entity_group']}]" + text[e["end"]:]
    return text

sample = "Ahmad from Petronas met Sarah in Kuala Lumpur on Monday."
print(detect(sample))
print(redact(sample))

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[{'entity_group': 'PER', 'score': np.float32(0.9849246), 'word': 'Ahmad', 'start': 0, 'end': 5}, {'entity_group': 'ORG', 'score': np.float32(0.98997784), 'word': 'Petronas', 'start': 11, 'end': 19}, {'entity_group': 'PER', 'score': np.float32(0.9973832), 'word': 'Sarah', 'start': 24, 'end': 29}, {'entity_group': 'LOC', 'score': np.float32(0.996169), 'word': 'Kuala Lumpur', 'start': 33, 'end': 45}]
[PER] from [ORG] met [PER] in [LOC] on Monday.
